In [1]:
import numpy as np 
import matplotlib
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import pandas as pd

from pathlib import Path
from skimage import io
import pickle
import dask.dataframe as dd
import json
import re
from ast import literal_eval

## Functions 

In [2]:
def find_bouts_array_adjusted(angles, fs, min_length=0.05, stepsize=10):
    """
    Finds bouts using tip angle data.

    Parameters
    ----------
    angles : numpy.ndarray
        The angle of the tip of the tail for all frames in continuously tracked data.
    fs : float
        The sampling frequency (frames per second).
    min_length : float, optional (default = 0.05)
        The minimum length of time a bout can last (seconds).
    stepsize : int, optional (default = 10)
        The step size you allow for bouts.

    Returns
    -------
    movement_frames : list
        A list of arrays where each array contains the frame numbers for a single bout.
    """
    def find_contiguous(data, minsize):
        runs = np.split(data, np.where(np.diff(data) > stepsize)[0] + 1)
        return [list(run) for run in runs if len(run) >= minsize]

    diffed = np.diff(angles)
    abs_diffed = np.abs(diffed)
    
    threshold = np.std(angles)
    above_threshold = np.where(abs_diffed > threshold)[0]

    movement_indices = find_contiguous(above_threshold, minsize=int(min_length * fs))

    # Adjust movement indices to start the bout 2 frames before and end 2 frames after
    adjusted_movement_indices = []
    for movement_index in movement_indices:
        start_frame = max(0, movement_index[0] - 1)
        end_frame = min(len(angles) - 1, movement_index[-1] + 1)
        adjusted_movement_indices.append((np.arange(start_frame, end_frame + 1)).tolist()) #have to put into list so that dataframe can be saved and loaded correctly

    return adjusted_movement_indices


def get_stimulus_name(file):
    """
    Gets the stimulus from the file name.
    
    Edit the function based on your file nomenclature. 
    
    e.g my files: 'movie_2023-05-26_2_cad_2.5_2_angles'
    
    """ 
    
    stimulus = []
    parts = file.split('movie_')[1].split('_angles.npy')  
    stimulus = '_'.join(parts[0].split('_')[1:])

    return stimulus

# funtion to match stim names with stim names in imagaing data 

def fix_stim_name(stim):
    
    """
    Matches the stimulus name in the IMG BHV Dataframe to the stim name in the Neural Dataframe.
    Clean up for my poor filenaming. 
    
    e.g cad_2.5 ---> cad_2.5mm 
    
    """ 
    if '_' in stim and '.' in stim:
        stim = stim + 'mm'
    elif '_' in stim:
        stim = stim + 'um'
    else:
        stim = stim

    return stim


In [27]:
## For finding bouts with running std window 

def running_std(trace, window_size_samples):
    """
    Calculate the running standard deviation over a trace and replace NaNs with zeros.
    
    Parameters:
    - trace (np.array or list): The input signal trace.
    - window_size_samples (int): The window size in number of samples.
    
    Returns:
    - np.array: The running standard deviation of the input trace with NaNs replaced by zeros.
    """
    # Ensure the input is a numpy array
    trace = np.asarray(trace)
    
    # Calculate the running standard deviation using pandas rolling window
    running_std_series = pd.Series(trace).rolling(window=window_size_samples).std()
    
    # Replace NaN values with zero
    running_std_series = running_std_series.fillna(0)
    
    # Convert to numpy array
    running_std = running_std_series.to_numpy()
    
    return running_std

def find_bouts(angles, fs, min_length=0.05, stepsize=10):
    """
    Finds bouts using tip angle data.

    Parameters
    ----------
    angles : numpy.ndarray
        The angle of the tip of the tail for all frames in continuously tracked data.
    fs : float
        The sampling frequency (frames per second).
    min_length : float, optional (default = 0.05)
        The minimum length of time a bout can last (seconds).
    stepsize : int, optional (default = 10)
        The step size you allow for bouts.

    Returns
    -------
    movement_frames : list
        A list of arrays where each array contains the frame numbers for a single bout.
    """
    def find_contiguous(data, minsize):
        runs = np.split(data, np.where(np.diff(data) > stepsize)[0] + 1)
        return [list(run) for run in runs if len(run) >= minsize]

    bhv_vigour = running_std(angles, 50)
    
    threshold = np.std(angles)
    above_threshold = np.where(bhv_vigour > threshold)[0]

    movement_indices = find_contiguous(above_threshold, minsize=int(min_length * fs))

    # Adjust movement indices to start the bout 2 frames before and end 2 frames after
    adjusted_movement_indices = []
    for movement_index in movement_indices:
        start_frame = max(0, movement_index[0] - 1)
        end_frame = min(len(angles) - 1, movement_index[-1]+1)
        adjusted_movement_indices.append((np.arange(start_frame, end_frame +1)).tolist()) #have to put into list so that dataframe can be saved and loaded correctly

    return adjusted_movement_indices




## Make DataFrame

In [29]:
## Using method from Portugues lab with running std and threshold 0.1 

folders_path = Path('/Volumes/LaCie/larval_HuC/behavior/7dpf_fb')

bhv_df = pd.DataFrame() 

for data_dir in folders_path.iterdir():
    print(data_dir)
    ## Load data from the folder 
    files = list((data_dir).glob('*_angles*'))
    fish_id = str(data_dir).split('/')[-1]
    
    fish_dict = [] 
    for file in files: 
        
        trial_name = str(file)
        stimulus = get_stimulus_name(trial_name)

        stim, trial_number = '_'.join(stimulus.split('_')[:-1]), int(stimulus.split('_')[-1])
        
        ## if stimulus name need to be fixed to match imaging stimulus name
        stim = fix_stim_name(stim)
        
        bhv_trace = np.load(file) # load the angles trace
        
        trace = bhv_trace[:6000] #cut the video to the same length as the imaging trials 
        trace_length = len(trace)

        # If the trace length is less than 6000, pad with zeros
        if trace_length < 6000:
            padding_length = 6000 - trace_length
            padding = np.zeros(padding_length)
            trace = np.concatenate([trace, padding])
            
        normalised_angles = trace - np.mean(trace) #normalise to zero 
        
        angles  = np.array(normalised_angles).tolist() # for saving and loading in dataframe

        trial_dict = {}

        sampling_frequency = 60.85  # sample frequency of behaviour camera 

        bouts = find_bouts(angles, sampling_frequency, min_length=0.05) # find the bout indices 
        

        trial_dict['angles'] = angles
        trial_dict['vigour'] = running_std(angles, 50)
        trial_dict['bouts'] = bouts
        trial_dict['fish_id'] = fish_id
        trial_dict['stimulus'] = stim 
        trial_dict['trial_number'] = str(trial_number)
        
        fish_dict.append(trial_dict)
    
    
    fish_df = pd.DataFrame(fish_dict)
    
    bhv_df = pd.concat([bhv_df, fish_df], ignore_index=True)

/Volumes/LaCie/orientation_exp/bhv/231207_1
/Volumes/LaCie/orientation_exp/bhv/231207_2
/Volumes/LaCie/orientation_exp/bhv/231208
/Volumes/LaCie/orientation_exp/bhv/231206_1
/Volumes/LaCie/orientation_exp/bhv/231206_2


## Save the DataFrame

In [41]:
bhv_df.to_csv(str(folders_path) + '_reorientation_df.csv')